# Remote Graph in LangGraph

## 什么是 Remote Graph？

Remote Graph 是 LangGraph 提供的一个客户端包装器，让你可以像调用本地 graph 一样调用远程部署的 LangGraph server。

**核心思想：** 你的 graph 部署在远程服务器上（Docker、Cloud 等），但在客户端代码中，你可以用和本地 graph 完全相同的 `.invoke()`、`.stream()` 接口来调用它。

## 为什么需要 Remote Graph？

| 方式 | 说明 |
|------|------|
| **本地 graph** | `graph.invoke(...)` — graph 代码和调用代码在同一个进程 |
| **LangGraph SDK** | `client.runs.create(...)` — 通过 REST API 调用远程 graph，API 风格 |
| **Remote Graph** | `remote_graph.invoke(...)` — 通过 REST API 调用远程 graph，但接口和本地 graph 一样 |

Remote Graph 的优势：
- 代码从本地切换到远程，几乎不需要改动
- 可以作为 sub-graph 嵌入到另一个本地 graph 中
- 支持 `.invoke()`、`.stream()`、`.get_state()`、`.update_state()` 等所有标准接口

## 基本用法

假设你的 LangGraph server 运行在 `http://localhost:8123`（比如通过 `docker compose up` 或 `langgraph dev`）。

In [ ]:
from langgraph.pregel.remote import RemoteGraph

# 连接到远程部署的 graph
remote_graph = RemoteGraph(
    "task_maistro",  # graph 的 ID（在 langgraph.json 中定义的）
    url="http://localhost:8123",  # LangGraph server 的地址
)

## 像本地 graph 一样调用

In [ ]:
from langchain_core.messages import HumanMessage

# invoke — 和本地 graph.invoke() 完全一样的接口
result = remote_graph.invoke({
    "messages": [HumanMessage(content="Add a task: finish the LangGraph course by Friday")]
})

for m in result["messages"]:
    m.pretty_print()

## 带 Thread 的调用（有状态）

Remote Graph 也支持 thread，实现多轮对话：

In [ ]:
# 使用 thread_id 保持对话状态
config = {"configurable": {"thread_id": "my-thread-1"}}

# 第一轮
result = remote_graph.invoke(
    {"messages": [HumanMessage(content="Add a task: buy groceries")]},
    config=config
)

# 第二轮 — 同一个 thread，graph 记得之前的对话
result = remote_graph.invoke(
    {"messages": [HumanMessage(content="What tasks do I have?")]},
    config=config
)

for m in result["messages"]:
    m.pretty_print()

## Streaming

In [ ]:
# stream — 和本地 graph.stream() 一样
for chunk in remote_graph.stream(
    {"messages": [HumanMessage(content="List all my tasks")]},
    config=config,
    stream_mode="updates"
):
    print(chunk)

## 作为 Sub-Graph 使用

Remote Graph 最强大的用法：嵌入到另一个本地 graph 中作为节点。

```python
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.pregel.remote import RemoteGraph

remote = RemoteGraph("task_maistro", url="http://localhost:8123")

# 本地 graph 把远程 graph 当作一个节点
builder = StateGraph(MessagesState)
builder.add_node("remote_agent", remote)
builder.add_edge(START, "remote_agent")
builder.add_edge("remote_agent", END)
graph = builder.compile()

# 调用本地 graph，它内部会调用远程 graph
result = graph.invoke({"messages": [HumanMessage(content="Hello")]})
```

这样你可以把多个部署在不同服务器上的 graph 组合成一个更大的系统。

## Remote Graph vs SDK Client 对比

| | Remote Graph | SDK Client |
|---|---|---|
| 接口风格 | `.invoke()` / `.stream()` (和本地一样) | `client.runs.create()` / `client.runs.stream()` (REST 风格) |
| 可作为 sub-graph | ✅ | ❌ |
| 需要手动管理 thread | 通过 config 传入 | 需要先 `client.threads.create()` |
| 适合场景 | 代码中直接调用、组合 graph | 前端/外部系统集成 |

## 总结

Remote Graph 让远程部署的 graph 在代码层面表现得和本地 graph 完全一样。你不需要关心 HTTP 请求、序列化这些细节 — 它全部封装好了。核心价值是**接口统一**和**可组合性**。